# 03 - U-Net 2D Baseline (Paper-Quality)

Trains and evaluates U-Net 2D baseline for slice interpolation.

## Pipeline
1. Train separate U-Net per sparse ratio (R=2, R=3, R=4)
2. Evaluate on ALL test cases
3. Report PSNR, SSIM, MAE, and ROI metrics
4. Save checkpoints for visualization notebook

## Important
- Full training requires significant GPU time
- Each R trains for 50 epochs on ~98 training volumes
- Set `FULL_EXPERIMENT = False` for quick debug

In [ ]:
# ========================== CONFIGURATION ==========================
FULL_EXPERIMENT = True
SPARSE_RATIOS = [2, 3, 4]
SEED = 42

# Override training params for debug
if not FULL_EXPERIMENT:
    MAX_TRAIN_VOLUMES = 4
    MAX_VAL_VOLUMES = 2
    NUM_EPOCHS = 3
    BATCH_SIZE = 4
    SPARSE_RATIOS = [2, 3]
else:
    MAX_TRAIN_VOLUMES = None  # Use all
    MAX_VAL_VOLUMES = None
    NUM_EPOCHS = 50
    BATCH_SIZE = 8
# ===================================================================

In [ ]:
import os, sys, json, subprocess, time
import numpy as np
import torch
from pathlib import Path

WORK_DIR = Path("/kaggle/working")
OUTPUT_ROOT = WORK_DIR / "outputs"
CHECKPOINT_ROOT = WORK_DIR / "checkpoints"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/PhoenixEvo/ct-slice-interpolation-3dgs.git"
REPO_DIR = WORK_DIR / "ct-slice-interpolation-3dgs"
if not (REPO_DIR / "src").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from src.utils.seed import set_seed
set_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Load split info and config
prepared_dir = OUTPUT_ROOT / "prepared"
with open(prepared_dir / "split_info.json") as f:
    split_info = json.load(f)

from src.utils.config import load_config, update_config
config = load_config("configs/default.yaml")

train_cases = split_info["train"]
val_cases = split_info["val"]
test_cases = split_info["test"]

if MAX_TRAIN_VOLUMES is not None:
    train_cases = train_cases[:MAX_TRAIN_VOLUMES]
if MAX_VAL_VOLUMES is not None:
    val_cases = val_cases[:MAX_VAL_VOLUMES]

print(f"Train: {len(train_cases)}, Val: {len(val_cases)}, Test: {len(test_cases)}")

In [ ]:
from tqdm import tqdm

def load_volumes_from_prepared(case_ids, prepared_dir):
    """Load preprocessed volumes from .npy files."""
    volumes = []
    loaded_ids = []
    for cid in tqdm(case_ids, desc="Loading volumes"):
        path = prepared_dir / f"volume_case{cid}.npy"
        if path.exists():
            volumes.append(np.load(path))
            loaded_ids.append(cid)
    return volumes, loaded_ids

print("Loading training volumes...")
train_volumes, train_ids = load_volumes_from_prepared(train_cases, prepared_dir)
print(f"Loaded {len(train_volumes)} train volumes")

print("Loading validation volumes...")
val_volumes, val_ids = load_volumes_from_prepared(val_cases, prepared_dir)
print(f"Loaded {len(val_volumes)} val volumes")

In [ ]:
from torch.utils.data import DataLoader
from src.data.dataset import UNetSliceDataset
from src.models.unet2d import UNet2D
from src.training.trainer_unet import TrainerUNet
from src.data.sparse_simulator import SparseSimulator
from src.evaluation.metrics import evaluate_volume

organ_labels = {"liver": 1, "bladder": 2, "lungs": 3, "kidneys": 4, "bone": 5}
all_unet_results = []

for R in SPARSE_RATIOS:
    print(f"\n{'='*60}")
    print(f"  Training U-Net for R={R}")
    print(f"{'='*60}")

    set_seed(SEED)

    # Build datasets
    train_ds = UNetSliceDataset(train_volumes, sparse_ratio=R, augment=True)
    val_ds = UNetSliceDataset(val_volumes, sparse_ratio=R, augment=False) if val_volumes else None

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2, pin_memory=True) if val_ds else None

    print(f"  Train samples: {len(train_ds)}, Val samples: {len(val_ds) if val_ds else 0}")

    # Create model
    model = UNet2D(
        in_channels=int(config.get("unet", {}).get("in_channels", 2)),
        out_channels=int(config.get("unet", {}).get("out_channels", 1)),
        features=list(config.get("unet", {}).get("features", [32, 64, 128, 256])),
    )
    print(f"  Parameters: {model.count_parameters():,}")

    ckpt_dir = str(CHECKPOINT_ROOT / f"unet_R{R}")

    trainer = TrainerUNet(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        lr=float(config.get("unet", {}).get("lr", 1e-4)),
        num_epochs=NUM_EPOCHS,
        scheduler_step=int(config.get("unet", {}).get("scheduler_step", 20)),
        scheduler_gamma=float(config.get("unet", {}).get("scheduler_gamma", 0.5)),
        mixed_precision=True,
        checkpoint_dir=ckpt_dir,
        device=device,
    )

    t_train_start = time.time()
    history = trainer.train()
    train_time = time.time() - t_train_start
    print(f"  Training time: {train_time:.1f}s")

    # ---- Evaluate on ALL test cases ----
    print(f"\n  Evaluating on {len(test_cases)} test cases...")

    for case_idx in tqdm(test_cases, desc=f"Eval R={R}"):
        vol_path = prepared_dir / f"volume_case{case_idx}.npy"
        if not vol_path.exists():
            continue

        volume = np.load(vol_path)
        labels = None
        lab_path = prepared_dir / f"labels_case{case_idx}.npy"
        if lab_path.exists():
            labels = np.load(lab_path)

        sim = SparseSimulator(sparse_ratio=R)
        sparse = sim.simulate(volume)
        pairs = sim.get_interpolation_pairs(volume)

        # Predict all target slices
        pred_slices = []
        t_infer_start = time.time()

        for p in pairs:
            before = torch.from_numpy(p["slice_before"]).float()
            after = torch.from_numpy(p["slice_after"]).float()
            for gt_slice in p["targets"]:
                pred = trainer.predict_slice(before, after).squeeze(0).numpy()
                pred = np.clip(pred, 0.0, 1.0)
                pred_slices.append(pred)

        infer_time = time.time() - t_infer_start

        if len(pred_slices) == 0:
            continue

        predictions = np.stack(pred_slices, axis=-1)
        gt_targets = sparse["target_slices"]

        n_eval = min(predictions.shape[2], gt_targets.shape[2])
        predictions = predictions[:, :, :n_eval]
        gt_targets = gt_targets[:, :, :n_eval]
        target_indices = sparse["target_indices"][:n_eval]

        ev = evaluate_volume(
            predictions=predictions,
            ground_truths=gt_targets,
            target_indices=target_indices,
            labels=labels,
            organ_labels=organ_labels,
        )
        ev["summary"]["inference_time_s"] = infer_time
        ev["summary"]["training_time_s"] = train_time

        all_unet_results.append({
            "method": "unet",
            "case_idx": int(case_idx),
            "sparse_ratio": R,
            **ev["summary"],
        })

    # Save per-R results
    out_dir = OUTPUT_ROOT / "unet_baseline" / f"R{R}"
    out_dir.mkdir(parents=True, exist_ok=True)
    with open(out_dir / "training_history.json", "w") as f:
        json.dump({k: [float(v) for v in vals] for k, vals in history.items()}, f, indent=2)

print(f"\nTotal U-Net evaluations: {len(all_unet_results)}")

In [ ]:
import pandas as pd

df_unet = pd.DataFrame(all_unet_results)

print("\n=== U-NET BASELINE - Summary Table ===")
summary = df_unet.groupby(["sparse_ratio"]).agg({
    "mean_psnr": ["mean", "std"],
    "mean_ssim": ["mean", "std"],
    "mean_mae": ["mean", "std"],
}).round(4)
print(summary.to_string())

# Save
df_unet.to_csv(OUTPUT_ROOT / "unet_baseline" / "summary.csv", index=False)
print("\nSaved to:", OUTPUT_ROOT / "unet_baseline" / "summary.csv")